In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

In [2]:
data = pd.read_csv("02SegundoTaller/data/imcv_modelo_limpio.csv")

In [3]:
data.head()

,IMCV,Ingresos,Salud,Trabajo,Capital
0,32.06,0.780000,2.550000,0.510000,3.310000
1,32.88,0.960000,2.620000,0.520000,3.680000
2,33.27,1.100000,2.650000,0.570000,3.830000
3,32.82,1.121182,2.569105,0.552032,3.739863
4,33.53,1.039000,3.041000,0.485000,3.667000


In [4]:
y = data["IMCV"].to_numpy()
predictoras = data.drop("IMCV", axis = 1)

In [8]:
design_matrix = np.column_stack((np.ones(len(data)), predictoras))

In [ ]:
n, p = design_matrix.shape
k = p - 1

array([11.54460366, -0.22524396,  2.06736101,  0.79852066,  4.58612383])

In [10]:
nombres = ["Intercepto", "Ingresos", "Salud", "Trabajo", "Capital"]

beta_hat = np.linalg.inv(design_matrix.T@design_matrix)@design_matrix.T@y

pd.Series(beta_hat, index = nombres)

Intercepto    11.544604
Ingresos      -0.225244
Salud          2.067361
Trabajo        0.798521
Capital        4.586124
dtype: float64

In [13]:
H = design_matrix@np.linalg.inv(design_matrix.T@design_matrix)@design_matrix.T

y_hat = H@y

y_hat[:6]

array([32.22799935, 34.03702173, 34.79535301, 34.1956175 , 34.80201861,
       36.14998058])

In [ ]:
residuals = (np.eye(n) - H)@y
residuals[:6]

#y-y_hat

array([-0.16799935, -1.15702173, -1.52535301, -1.3756175 , -1.27201861,
       -1.37998058])

In [15]:
sigma_hat = np.sum(residuals**2)/(n-p)
sigma_hat

np.float64(1.8923423480690407)

In [16]:
model = smf.ols("IMCV~Ingresos + Salud + Trabajo + Capital", data=data).fit()

In [17]:
model.params

Intercept    11.544604
Ingresos     -0.225244
Salud         2.067361
Trabajo       0.798521
Capital       4.586124
dtype: float64

In [18]:
model.fittedvalues[:6]

0    32.227999
1    34.037022
2    34.795353
3    34.195617
4    34.802019
5    36.149981
dtype: float64

In [19]:
model.resid[:6]

0   -0.167999
1   -1.157022
2   -1.525353
3   -1.375617
4   -1.272019
5   -1.379981
dtype: float64

In [20]:
model.mse_resid

np.float64(1.8923423480692148)

In [ ]:
SST = np.sum((y-y.mean())**2)
SSR = np.sum((y_hat - y.mean())**2)
SSE = np.sum(residuals**2)

MSR = SSR/k
MSE = SSE/(n-p)

In [24]:
F0 = MSR/MSE

F0

np.float64(2814.1048408581405)

In [ ]:
stats.f.ppf(0.95, k, n-p)

np.float64(2.4271155491611367)

In [26]:
stats.f.sf(F0, k, n-p)

np.float64(3.203231793136678e-149)

In [27]:
reduced_model = smf.ols("IMCV~1", data=data).fit()
full_model = model

SSE_R = np.sum(reduced_model.resid**2)
SSE_F = np.sum(full_model.resid**2)

In [28]:
SS_extra = SSE_R - SSE_F
gl_extra = reduced_model.df_resid - full_model.df_resid

F0_extra = (SS_extra/gl_extra)/(SSE_F/(n-p))

F0_extra

np.float64(2814.1048408582296)

In [29]:
stats.f.sf(F0_extra, gl_extra, n-p)

np.float64(3.2032317931285533e-149)

In [31]:
anova_lm(reduced_model, full_model)

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,167.0,21609.450852,0.0,NaN,NaN,NaN
1,163.0,308.451803,4.0,21300.999049,2814.104841,3.203232e-149


In [32]:
XtX_inv = np.linalg.inv(design_matrix.T@design_matrix)

var_beta = sigma_hat * XtX_inv

ee_beta = np.sqrt(np.diag(var_beta))

estadistico_t = beta_hat/ee_beta
valor_p = 2*stats.t.sf(np.abs(estadistico_t), n-p)
cuantil_t = stats.t.ppf(0.975, n-p)

In [36]:
IC = np.column_stack((beta_hat - cuantil_t*ee_beta, beta_hat + cuantil_t*ee_beta))


pd.DataFrame(
    {
        "Estimacion": beta_hat,
        "ee": ee_beta,
        "T0": estadistico_t,
        "valor_p": valor_p,
        "LI" : IC[:,0],
        "LS" : IC[:, 1]
    },
    index=nombres

)

,Estimacion,ee,T0,valor_p,LI,LS
Intercepto,11.544604,1.331458,8.670648,4.126062e-15,8.915474,14.173733
Ingresos,-0.225244,0.771548,-0.291938,7.707055e-01,-1.748762,1.298274
Salud,2.067361,0.464132,4.454248,1.557449e-05,1.150874,2.983848
Trabajo,0.798521,1.315071,0.607207,5.445583e-01,-1.798251,3.395292
Capital,4.586124,0.232107,19.758669,4.028380e-45,4.127800,5.044448


In [37]:
model.conf_int()

,0,1
Intercept,8.915474,14.173733
Ingresos,-1.748762,1.298274
Salud,1.150874,2.983848
Trabajo,-1.798251,3.395292
Capital,4.127800,5.044448


In [40]:
R2 = SSR/SST
R2_adj = 1-(n-1)*MSE/SST

In [41]:
R2

np.float64(0.9857260693548614)

In [42]:
R2_adj

np.float64(0.9853757888483586)

In [47]:
modelo_reducido = smf.ols("IMCV~Salud+Trabajo+Capital", data = data).fit()

MSE_reducido = np.sum(modelo_reducido.resid**2)/modelo_reducido.df_resid

R2_adj_reducido = 1-(n-1)*MSE_reducido/SST

In [45]:
modelo_reducido.rsquared

np.float64(0.9857186059605418)

In [48]:
R2_adj_reducido

np.float64(0.9854573609476249)

In [50]:
modelo_B = smf.ols("IMCV~Ingresos + Capital", data=data).fit()

tabla_extra = anova_lm(modelo_B, model)
tabla_extra

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,165.0,346.015528,0.0,NaN,NaN,NaN
1,163.0,308.451803,2.0,37.563726,9.925193,0.000086


In [51]:
modelo_sin_trabajo = smf.ols("IMCV~Ingresos + Salud + Capital", data=data).fit()

tabla_trabajo = anova_lm(modelo_sin_trabajo, model)
tabla_trabajo

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,164.0,309.149511,0.0,NaN,NaN,NaN
1,163.0,308.451803,1.0,0.697708,0.368701,0.544558


In [ ]:
model.tvalues["Trabajo"]**2

np.float64(0.3687006636202428)

In [55]:
model.pvalues["Trabajo"]

np.float64(0.5445582910942276)